# Credit Card Fraud Detection using Logistic Regression

โน้ตบุ๊คนี้ใช้โมเดล **Logistic Regression** เพื่อตรวจจับการทุจริตบัตรเครดิต
โดยมีขั้นตอน ได้แก่:
1. นำเข้าข้อมูลและสำรวจข้อมูลเบื้องต้น (EDA)
2. เตรียมข้อมูล (Preprocessing)
3. จัดการ Class Imbalance ด้วย Under-sampling
4. แบ่งข้อมูล Train/Test
5. สร้างและฝึกโมเดล Logistic Regression
6. ประเมินผลโมเดล (Accuracy, Confusion Matrix, ROC Curve)

## 1. Import Libraries

In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

## 2. Load Data

In [ ]:
# Mount Drive and Read creditcard
from google.colab import drive
drive.mount('/content/drive')
credit_card_data = pd.read_csv('creditcard.csv')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ดูชื่อคอลัมน์ทั้งหมด
credit_card_data.keys()

In [ ]:
# ข้อมูลเบื้องต้นของ Dataset
credit_card_data.info()

In [ ]:
# ตรวจสอบค่า Missing Values
credit_card_data.isnull().sum()

In [ ]:
# ดูการกระจายของ Class (0 = ปกติ, 1 = ทุจริต)
credit_card_data['Class'].value_counts()

In [ ]:
# แสดงสถิติเบื้องต้น
credit_card_data.describe()

## 4. Data Preprocessing

In [ ]:
# ลบคอลัมน์ Time ออก (ไม่ใช้ในการทำนาย)
credit_card_data = credit_card_data.drop('Time', axis=1)

In [ ]:
# Standard Scaling สำหรับคอลัมน์ Amount
from sklearn import preprocessing
scaler = preprocessing.StandardScaler()

credit_card_data['std_Amount'] = scaler.fit_transform(
    credit_card_data['Amount'].values.reshape(-1, 1)
)

# ลบคอลัมน์ Amount เดิมออก
credit_card_data = credit_card_data.drop('Amount', axis=1)

print("Shape หลังการเตรียมข้อมูล:", credit_card_data.shape)

In [ ]:
# แสดง Class Distribution ก่อน Under-sampling
sns.countplot(x='Class', data=credit_card_data)
plt.title('Class Distribution (ก่อน Under-sampling)')
plt.xlabel('Class (0 = ปกติ, 1 = ทุจริต)')
plt.ylabel('Count')
plt.show()

## 5. Handle Class Imbalance with Under-sampling

In [ ]:
import imblearn
from imblearn.under_sampling import RandomUnderSampler

# กำหนดสัดส่วน 0.5 หมายความว่า Class 1 จะมี 50% ของ Class 0
undersample = RandomUnderSampler(sampling_strategy=0.5)

In [ ]:
# กำหนด Features และ Target
cols = credit_card_data.columns.tolist()
cols = [c for c in cols if c not in ['Class']]
target = 'Class'

# ลบแถวที่มีค่า null ใน Class
credit_card_data.dropna(subset=['Class'], inplace=True)

X = credit_card_data[cols]
Y = credit_card_data[target]

# ทำ Under-sampling
X_under, Y_under = undersample.fit_resample(X, Y)

print(f"ขนาดข้อมูลหลัง Under-sampling: X={X_under.shape}, Y={Y_under.shape}")
print(f"Class Distribution: {dict(Y_under.value_counts())}")

In [ ]:
# เปรียบเทียบ Class Distribution ก่อนและหลัง Under-sampling
from pandas import DataFrame
df_under = DataFrame(Y_under, columns=['Class'])

fig, axs = plt.subplots(ncols=2, figsize=(12, 5))
sns.countplot(x='Class', data=credit_card_data, ax=axs[0])
sns.countplot(x='Class', data=df_under, ax=axs[1])

axs[0].set_title('Original Data')
axs[1].set_title('Undersampled Data')
plt.tight_layout()
plt.show()

## 6. Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_under, Y_under,
    test_size=0.2,
    random_state=100,
    stratify=Y_under  # เพื่อรักษาสัดส่วนของ Class ในทั้ง Train และ Test
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"\nTraining Class Distribution:\n{y_train.value_counts()}")
print(f"\nTest Class Distribution:\n{y_test.value_counts()}")

## 7. Logistic Regression Model

In [ ]:
# Import Logistic Regression และเครื่องมือประเมินผล
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    classification_report
)

In [ ]:
# สร้างและฝึกโมเดล Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000,       # จำนวนรอบสูงสุดในการ Optimize
    random_state=42,     # กำหนด Seed เพื่อให้ผลลัพธ์คงที่
    class_weight='balanced',  # ช่วยจัดการ Class Imbalance เพิ่มเติม
    solver='lbfgs'       # Algorithm สำหรับ Optimization
)

lr_model.fit(X_train, y_train)
print("✅ ฝึกโมเดล Logistic Regression เสร็จสิ้น")

## 8. Model Evaluation

In [ ]:
# ทำนายผล
y_pred_lr = lr_model.predict(X_test)
y_pred_proba_lr = lr_model.predict_proba(X_test)[:, 1]  # ความน่าจะเป็นของ Class 1

# แสดงผลการประเมิน
print("=" * 50)
print("📊 ผลการประเมินโมเดล Logistic Regression")
print("=" * 50)
print(f"\n🎯 Accuracy Score: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"\n📈 ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_lr):.4f}")
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_lr,
                             target_names=['Normal (0)', 'Fraud (1)']))

In [ ]:
# แสดง Confusion Matrix
cm = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Normal', 'Predicted Fraud'],
            yticklabels=['Actual Normal', 'Actual Fraud'])
plt.title('Confusion Matrix - Logistic Regression', fontsize=14)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

# สรุป Confusion Matrix
tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negative  (TN): {tn}  ← ทำนายปกติถูก")
print(f"False Positive (FP): {fp}  ← ทำนายว่าทุจริต แต่จริงๆ ปกติ")
print(f"False Negative (FN): {fn}  ← ทำนายว่าปกติ แต่จริงๆ ทุจริต")
print(f"True Positive  (TP): {tp}  ← ทำนายทุจริตถูก")

In [ ]:
# แสดง ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_lr)
auc_lr = roc_auc_score(y_test, y_pred_proba_lr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2,
         label=f'ROC curve - Logistic Regression (AUC = {auc_lr:.4f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--', label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.1, color='blue')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.title('ROC Curve - Logistic Regression', fontsize=14)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# แสดง Precision-Recall Curve
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_pred_proba_lr)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='green', lw=2,
         label='Precision-Recall Curve')
plt.fill_between(recall, precision, alpha=0.1, color='green')
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curve - Logistic Regression', fontsize=14)
plt.legend(loc='lower left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Feature Importance (Coefficients)

In [ ]:
# แสดง Feature Importance จาก Coefficients ของ Logistic Regression
feature_names = X_train.columns.tolist()
coefficients = lr_model.coef_[0]

# สร้าง DataFrame สำหรับแสดงผล
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients)
}).sort_values('Abs_Coefficient', ascending=False)

print("Top 10 Features ที่มีผลต่อการทำนายมากที่สุด:")
print(feature_importance_df.head(10).to_string(index=False))

# Plot Feature Importance
plt.figure(figsize=(10, 8))
top_features = feature_importance_df.head(15)
colors = ['red' if c < 0 else 'blue' for c in top_features['Coefficient']]
plt.barh(top_features['Feature'], top_features['Coefficient'], color=colors)
plt.xlabel('Coefficient Value', fontsize=12)
plt.title('Top 15 Feature Importance (Logistic Regression Coefficients)', fontsize=13)
plt.axvline(x=0, color='black', linestyle='-', lw=0.8)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 10. Cross-Validation

In [ ]:
# ประเมินโมเดลด้วย Cross-Validation เพื่อให้ผลลัพธ์น่าเชื่อถือมากขึ้น
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_accuracy = cross_val_score(lr_model, X_under, Y_under,
                               cv=cv, scoring='accuracy')
cv_auc = cross_val_score(lr_model, X_under, Y_under,
                          cv=cv, scoring='roc_auc')
cv_f1 = cross_val_score(lr_model, X_under, Y_under,
                         cv=cv, scoring='f1')

print("📊 ผลการทำ 5-Fold Cross-Validation:")
print("-" * 45)
print(f"Accuracy : {cv_accuracy.mean():.4f} ± {cv_accuracy.std():.4f}")
print(f"ROC-AUC  : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
print(f"F1-Score : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")

## 11. สรุปผลการวิเคราะห์

| Metric | ผลลัพธ์ |
|--------|----------|
| **Algorithm** | Logistic Regression |
| **Solver** | LBFGS |
| **Class Weight** | Balanced |
| **Sampling Strategy** | Under-sampling (ratio 0.5) |
| **Train/Test Split** | 80% / 20% |

### ข้อดีของ Logistic Regression สำหรับงานนี้:
- ✅ **ตีความง่าย**: Coefficients บอกทิศทางและขนาดของผลต่อการทำนาย
- ✅ **เร็ว**: ฝึกและทำนายได้รวดเร็ว
- ✅ **Probabilistic Output**: ได้ความน่าจะเป็นในการเป็น Fraud
- ✅ **รองรับ Regularization**: ป้องกัน Overfitting ได้

### ข้อจำกัด:
- ⚠️ สมมติว่าความสัมพันธ์ระหว่าง Features และ Target เป็น Linear
- ⚠️ อาจมีประสิทธิภาพต่ำกว่า SVM หรือ Tree-based Models สำหรับข้อมูลที่ซับซ้อน